## Imports

In [1]:
import sqlite3
import glob
import time
import itertools 

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

/var/folders/m8/kq3tbvsx79n47wprd2zwv5hm0000gn/T/ipykernel_18879/1025221560.py:7: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


_____

## 1. Data Ingestion 

Read the data from the database.

In [2]:
annotated_dbs = glob.glob(f'./dataset/pacheco_system/*.db')

print("Found the following dbs : " , annotated_dbs)

synchronous_annotators = ['group_1' , 'group_2'] 
asynchronous_annotators = ['async_1' , 'async_2' , 'async_3'] 
annotators = synchronous_annotators + asynchronous_annotators
annotator2df = {}

for db_path in  annotated_dbs : 

    con = sqlite3.connect(db_path)
    tweet_df = pd.read_sql_query('SELECT * FROM tweet;' , con) 
    theme_df = pd.read_sql_query('SELECT * FROM theme;' , con) 
    theme_df = theme_df.rename(columns={'id' : 'theme_id'})
    merged_df = tweet_df.merge(theme_df , how='left' , on='theme_id')

    for key in annotators : 
        if key in db_path : 
            annotator2df[key] = merged_df

Found the following dbs :  ['./dataset/pacheco_system/async_1.db', './dataset/pacheco_system/group_1.db', './dataset/pacheco_system/async_3.db', './dataset/pacheco_system/group_2.db', './dataset/pacheco_system/async_2.db']


### 1.a. Random Sampling rows for manual annotations

Pull 200 samples from both synchronous and asynchronous groups to manually review label quality. Pulling from top 25th percentile is done later.

In [3]:
synchronous_random_samples = []
asynchronous_random_samples = []

for annotator , df in annotator2df.items(): 
    if annotator in synchronous_annotators: 
        synchronous_random_samples.append(df.sample(n=100))
    if annotator in asynchronous_annotators: 
        asynchronous_random_samples.append(df.sample(n=67))

res_df = pd.concat(synchronous_random_samples)
async_df = pd.concat(asynchronous_random_samples)

res_df = res_df[['text' , 'name']].sample(frac=1)
async_df = async_df[['text' , 'name']].sample(frac=1)

res_df.to_csv('./dataset/generated_samples/pacheco_sync_sample_all.csv' , index=False, sep='\t')
async_df.to_csv('./dataset/generated_samples/pacheco_async_sample_all.csv' , index=False, sep='\t')

_____

## 2. Jaccard Similarity 

Jaccard similarity for two themes is calculated by the union of their documents divided by the intersection of their documents.

In [4]:
results = []

for anno_1 , anno_2 in itertools.permutations(annotators , 2): 
    anno_1_themes = annotator2df[anno_1]['name'].unique()
    anno_2_themes = annotator2df[anno_2]['name'].unique()
    for anno_1_theme , anno_2_theme in itertools.product(anno_1_themes , anno_2_themes): 
        result = {'anno_1' : anno_1 , 
                  'anno_2' : anno_2 , 
                  'anno_1_theme' : anno_1_theme ,
                  'anno_2_theme' : anno_2_theme}
        anno_1_tweet_ids = set(annotator2df[anno_1][annotator2df[anno_1]['name']==anno_1_theme]['tweet_id'])
        anno_2_tweet_ids = set(annotator2df[anno_2][annotator2df[anno_2]['name']==anno_2_theme]['tweet_id'])
        intersection = anno_1_tweet_ids.intersection(anno_2_tweet_ids)
        union = anno_1_tweet_ids.union(anno_2_tweet_ids)
        jaccard_sim = len(intersection) / len(union)
        result['jaccard_sim'] = jaccard_sim
        results.append(result)

jaccard_df = pd.DataFrame(results)

### 2.a. Getting max jaccard similarity for synchronous experiments

In [5]:
max_jacc_sims = []
filtered_df = jaccard_df[(jaccard_df['anno_1'].isin(synchronous_annotators)) 
                         &(jaccard_df['anno_2'].isin(synchronous_annotators))]

for anno_1_theme in filtered_df['anno_1_theme'].unique(): 
    if ('kmeans' not in anno_1_theme.lower()) and ('Unknown' not in anno_1_theme.strip()) and (anno_1_theme != "None"):
        theme_filtered_df  = filtered_df[(filtered_df['anno_1_theme'] == anno_1_theme) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('None')) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('Kmeans')) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('Unknown'))]
        max_jacc_sims.append(theme_filtered_df.loc[(theme_filtered_df['jaccard_sim'].idxmax())].to_dict())
res_df = pd.DataFrame(max_jacc_sims)

print("Synchronous Jaccard Similarity")
print(f"Average Max Jaccard Similarity: {res_df['jaccard_sim'].mean():.2f}")
print(f"Standard Deviation of Jaccard Similarity: {res_df['jaccard_sim'].std():.2f}")
print(res_df.count())

Synchronous Jaccard Similarity
Average Max Jaccard Similarity: 0.27
Standard Deviation of Jaccard Similarity: 0.17
anno_1          41
anno_2          41
anno_1_theme    41
anno_2_theme    41
jaccard_sim     41
dtype: int64


In [6]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,jaccard_sim
0,group_1,group_2,DonationMatchingForClimate,ClimateChangeMovements,0.429577
1,group_1,group_2,AbandonedWells,RenewableEnergyPromotionAndGrowth,0.071592
2,group_1,group_2,GreenEnergyIsCostEffective,RenewableEnergyPromotionAndGrowth,0.042781
3,group_1,group_2,ClimateEmergency,ClimateChangeMovements,0.230769
4,group_1,group_2,NuclearRevenueSupportsJobs,CarbonEmissionReduction,0.184932
5,group_1,group_2,HazardsOfWells,LawmakersAndOilAndGas,0.121451
6,group_1,group_2,VirginiaConservatives,CarbonEmissionReduction,0.035354
7,group_1,group_2,DemandClimateActionFromLeaders,ClimateChangeMovements,0.057143
8,group_1,group_2,PublicLandExploitation,LawmakersAndOilAndGas,0.182825
9,group_1,group_2,GreenEnergyCreatesJobs,RenewableEnergyEconomicBenefits,0.258359


### 2.b. Getting max jaccard similarity for asynchronous experiments

In [7]:
max_jacc_sims = []
filtered_df = jaccard_df[(jaccard_df['anno_1'].isin(asynchronous_annotators)) 
                         &(jaccard_df['anno_2'].isin(asynchronous_annotators))]
for anno_1_theme in filtered_df['anno_1_theme'].unique():
    if ('kmeans' not in anno_1_theme.lower()) and  ('Unknown' not in anno_1_theme.strip()) and (anno_1_theme != "None"):
        theme_filtered_df  = filtered_df[(filtered_df['anno_1_theme'] == anno_1_theme) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('None')) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('Kmeans')) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('Unknown'))]
        max_jacc_sims.append(theme_filtered_df.loc[(theme_filtered_df['jaccard_sim'].idxmax())].to_dict())
res_df = pd.DataFrame(max_jacc_sims)

print("Asynchronous Jaccard Similarity")
print(f"Average Max Jaccard Similarity: {res_df['jaccard_sim'].mean():.2f}")
print(f"Standard Deviation of Jaccard Similarity: {res_df['jaccard_sim'].std():.2f}")

Asynchronous Jaccard Similarity
Average Max Jaccard Similarity: 0.33
Standard Deviation of Jaccard Similarity: 0.22


In [8]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,jaccard_sim
0,async_1,async_3,AdvocateForGreenEnergy,LegislatorsMustActAgainstClimateChange,0.203361
1,async_1,async_2,GreenPowerExpensive,PoliticalActivismForOilAndGas,0.433511
2,async_1,async_3,ClimateActionDonation,OilAndGasJobs,0.495468
3,async_1,async_2,GreenEnergyWorks,SupportForOilAndGas,0.138889
4,async_1,async_2,AdvocateForGreenEnergyPoliticalAction,NeedQuickResponseForClimateCrisis,0.132964
5,async_1,async_3,AntiTax,VirginiaConservativeCampaign,0.292576
6,async_1,async_3,GreenEnergySavesMoney,RenewableEnergyInfrastructureGrowth,0.261346
7,async_1,async_3,StopOilPollution,SouthwestGasInvestment,0.148936
8,async_1,async_2,DemocratOutreach,SupportForBuildBackBetterAct,0.660777
9,async_1,async_3,OilCreatesJobs,OilAndGasTax,0.686084


______

## 3. Centroid Cosine Similarity 

### 3.a. Loading SBERT Vectors 

In [9]:
sbert_vectors = np.load('./dataset/sbert.npy')

### 3.b. Calculating centroids for sync + async experiments

In [10]:
results = []

for annotator, df in annotator2df.items(): 
    cosine_sims = []
    themes = []
    for i , theme in enumerate(df['name'].unique()): 
        if 'kmeans' not in theme.lower() and theme.lower() != "None": 
            result = {'annotator' : annotator}
            ids = df[df['name'] == theme]['tweet_id'].tolist()
            result['theme'] = theme
            theme_vectors = sbert_vectors[ids]
            theme_centroid = np.average(theme_vectors, axis=0, keepdims=True)

            result['theme_centroid'] = theme_centroid
            result['theme_vectors'] = theme_vectors

            dot_prod = np.dot(theme_centroid , theme_vectors.T).squeeze(0)
            dot_prod = np.expand_dims(dot_prod , axis=-1)

            norm = np.linalg.norm(theme_vectors , axis=1, keepdims=True)
            cosine_sim = (dot_prod/norm)
            result['cosine_sim']  = cosine_sim 
            results.append(result)


### 3.c. Calculating synchronous centroid cosine similarity

In [11]:
cosine_sim_results = []

for anno_1 , anno_2 in itertools.permutations(synchronous_annotators , 2):
    anno_1_results = [r for r in results if r['annotator']==anno_1]
    anno_2_results = [r for r in results if r['annotator']==anno_2]
    for anno_1_result in anno_1_results: 
        for anno_2_result in anno_2_results:
            if ('kmeans' not in anno_1_result['theme'].lower()) and  ('kmeans' not in anno_2_result['theme'].lower()) and (anno_2_result['theme'] != "None"): 
                cosine_sim = cosine_similarity(anno_1_result['theme_centroid'] , anno_2_result['theme_centroid'])
                cosine_sim_result = {'anno_1' : anno_1 , 
                                    'anno_2' : anno_2 , 
                                    'anno_1_theme' : anno_1_result['theme'] , 
                                    'anno_2_theme' : anno_2_result['theme'] , 
                                    'cosine_sim' : cosine_sim.squeeze()}
                cosine_sim_results.append(cosine_sim_result)
                
cosine_sim_df = pd.DataFrame(cosine_sim_results)

In [12]:
max_cosine_sims = []

for anno_1 , anno_2 in itertools.permutations(synchronous_annotators , 2):
    filtered_df = cosine_sim_df[(cosine_sim_df['anno_1']== anno_1) & (cosine_sim_df['anno_2']== anno_2)]
    for anno_1_theme in filtered_df['anno_1_theme'].unique():
        if ('kmeans' not in anno_1_theme.lower()) and ('unknown' not in anno_1_theme.lower()) and (anno_1_theme != "None"):
            theme_filtered_df  = filtered_df[(filtered_df['anno_1_theme'] == anno_1_theme) &
                                            ~(filtered_df['anno_2_theme'].str.contains('None')) & 
                                              ~(filtered_df['anno_2_theme'].str.contains('Kmeans')) &
                                              ~(filtered_df['anno_2_theme'].str.contains('Unknown'))]
            max_cosine_sims.append(theme_filtered_df.loc[(theme_filtered_df['cosine_sim'].idxmax())].to_dict())
 
res_df = pd.DataFrame(max_cosine_sims)


print("Synchronous Centroid Cosine Similarity")
print(f"Average Max Centroid Cosine Similarity: {res_df['cosine_sim'].mean():.2f}")
print(f"Standard Deviation of Centroid Cosine Similarity: {res_df['cosine_sim'].std():.2f}")
print(res_df.count())

Synchronous Centroid Cosine Similarity
Average Max Centroid Cosine Similarity: 0.95
Standard Deviation of Centroid Cosine Similarity: 0.05
anno_1          41
anno_2          41
anno_1_theme    41
anno_2_theme    41
cosine_sim      41
dtype: int64


In [13]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,cosine_sim
0,group_1,group_2,DonationMatchingForClimate,ClimateChangeMovements,0.9921806
1,group_1,group_2,AbandonedWells,CarbonEmissionReduction,0.93390083
2,group_1,group_2,GreenEnergyIsCostEffective,CarbonEmissionReduction,0.9304381
3,group_1,group_2,ClimateEmergency,ClimateChangeMovements,0.9816895
4,group_1,group_2,NuclearRevenueSupportsJobs,CarbonEmissionReduction,0.9464816
5,group_1,group_2,HazardsOfWells,CarbonEmissionReduction,0.93027365
6,group_1,group_2,VirginiaConservatives,CarbonEmissionReduction,0.8829772
7,group_1,group_2,DemandClimateActionFromLeaders,ClimateChangeMovements,0.9109599
8,group_1,group_2,PublicLandExploitation,LawmakersAndOilAndGas,0.9677283
9,group_1,group_2,GreenEnergyCreatesJobs,RenewableEnergyEconomicBenefits,0.96966934


### 3.d. Calculating asynchronous centroid cosine similarity

In [14]:
cosine_sim_results = []

for anno_1 , anno_2 in itertools.permutations(asynchronous_annotators , 2):
    anno_1_results = [r for r in results if r['annotator']==anno_1]
    anno_2_results = [r for r in results if r['annotator']==anno_2]
    for anno_1_result in anno_1_results : 
        for anno_2_result in anno_2_results :
            if ('kmeans' not in anno_1_result['theme'].lower()) and  ('kmeans' not in anno_2_result['theme'].lower()): 
                cosine_sim = cosine_similarity(anno_1_result['theme_centroid'] , anno_2_result['theme_centroid'])
                cosine_sim_result = {'anno_1' : anno_1 , 
                                    'anno_2' : anno_2 , 
                                    'anno_1_theme' : anno_1_result['theme'] , 
                                    'anno_2_theme' : anno_2_result['theme'] , 
                                    'cosine_sim' : cosine_sim.squeeze()}
                cosine_sim_results.append(cosine_sim_result)
                
cosine_sim_df = pd.DataFrame(cosine_sim_results)

In [16]:
max_cosine_sims = []

for anno_1 , anno_2 in itertools.permutations(asynchronous_annotators , 2):
    filtered_df = cosine_sim_df[(cosine_sim_df['anno_1']== anno_1) & (cosine_sim_df['anno_2']== anno_2)]
    for anno_1_theme in filtered_df['anno_1_theme'].unique():
        if ('kmeans' not in anno_1_theme.lower()) and ('unknown' not in anno_1_theme.lower()):
            theme_filtered_df  = filtered_df[(filtered_df['anno_1_theme'] == anno_1_theme) &
                                            ~(filtered_df['anno_2_theme'].str.contains('None')) & 
                                              ~(filtered_df['anno_2_theme'].str.contains('Kmeans')) &
                                              ~(filtered_df['anno_2_theme'].str.contains('Unknown'))]
            max_cosine_sims.append(theme_filtered_df.loc[(theme_filtered_df['cosine_sim'].idxmax())].to_dict())
 
res_df = pd.DataFrame(max_cosine_sims)

print("Asynchronous Centroid Cosine Similarity")
print(f"Average Max Centroid Cosine Similarity: {res_df['cosine_sim'].mean():.2f}")
print(f"Standard Deviation of Centroid Cosine Similarity: {res_df['cosine_sim'].std():.2f}")
print(res_df.count())

Asynchronous Centroid Cosine Similarity
Average Max Centroid Cosine Similarity: 0.94
Standard Deviation of Centroid Cosine Similarity: 0.09
anno_1          88
anno_2          88
anno_1_theme    88
anno_2_theme    88
cosine_sim      88
dtype: int64


In [17]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,cosine_sim
0,async_1,async_2,AdvocateForGreenEnergy,NeedQuickResponseForClimateCrisis,0.9561469
1,async_1,async_2,GreenPowerExpensive,PoliticalActivismForOilAndGas,0.97955346
2,async_1,async_2,ClimateActionDonation,DonateToSupportClimateChangeActivism,0.9623673
3,async_1,async_2,GreenEnergyWorks,ChallengesFacingRenewableEnergy,0.9386797
4,async_1,async_2,AdvocateForGreenEnergyPoliticalAction,NeedQuickResponseForClimateCrisis,0.96940005
...,...,...,...,...,...
83,async_3,async_2,RenewableEnergyInfrastructureGrowth,InroadsMadeByRenewableEnergy,0.9530623
84,async_3,async_2,NuclearEnergyEducation,InroadsMadeByNuclearEnergy,0.94540393
85,async_3,async_2,OilAndGasTax,EconomicAdvantagesOfOilAndGas,0.99593884
86,async_3,async_2,BuildBetterAct,InroadsMadeByNuclearEnergy,0.97741723


____

## 4. Group Average Cosine Similarity

In [18]:
results = {}
annotators = []

for annotator, df in annotator2df.items():
  results[annotator] = {}
  annotators.append(annotator)
  for i , theme in enumerate(df['name'].unique()):
      if ('kmeans' not in theme.lower()) and ('unknown' not in theme.lower()):
          ids = df[df['name'] == theme]['tweet_id'].tolist()
          theme_vectors = sbert_vectors[ids]
          results[annotator][theme] = theme_vectors

### 4.a. Calculating Group Average Similarities

In [19]:
global_average_sims = []

for (anno1 , anno2) in tqdm(itertools.combinations_with_replacement(annotators, 2)):
  anno1_themes = results[anno1]
  anno2_themes = results[anno2]
  for anno1_theme, anno1_theme_vectors in anno1_themes.items():
    for anno2_theme, anno2_theme_vectors in anno2_themes.items():
      s_time = time.time()
      cosine_sims = cosine_similarity(anno1_theme_vectors , anno2_theme_vectors)
      average_sim = np.average(cosine_sims)
      std_deviation = np.std(cosine_sims)
      global_average_sims.append({'anno1' : anno1 ,
                                  'anno2' : anno2 ,
                                  'anno1_theme' : anno1_theme ,
                                  'anno2_theme' : anno2_theme ,
                                  'average_sim' : average_sim ,
                                  'std_deviation' : std_deviation
                                  })

0it [00:00, ?it/s]

In [20]:
global_average_sims

[{'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AdvocateForGreenEnergy',
  'anno2_theme': 'AdvocateForGreenEnergy',
  'average_sim': 0.3770807,
  'std_deviation': 0.15226154},
 {'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AdvocateForGreenEnergy',
  'anno2_theme': 'GreenPowerExpensive',
  'average_sim': 0.29164362,
  'std_deviation': 0.12073554},
 {'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AdvocateForGreenEnergy',
  'anno2_theme': 'ClimateActionDonation',
  'average_sim': 0.33285764,
  'std_deviation': 0.13983954},
 {'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AdvocateForGreenEnergy',
  'anno2_theme': 'GreenEnergyWorks',
  'average_sim': 0.3009228,
  'std_deviation': 0.12352273},
 {'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AdvocateForGreenEnergy',
  'anno2_theme': 'AdvocateForGreenEnergyPoliticalAction',
  'average_sim': 0.3840152,
  'std_deviation': 0.15012325},
 {'anno1': 'async_1',
  'anno2': 'async_1',

In [21]:
global_average_df = pd.DataFrame(global_average_sims)

### 4.b. Calculating Synchronous Group Average Cosine Similarities

In [22]:
max_cosine_sims = []

for anno_1 , anno_2 in itertools.permutations(synchronous_annotators , 2):
    filtered_df = global_average_df[(global_average_df['anno1'] == anno_1)&
                                    (global_average_df['anno2'] == anno_2)]
    for anno_1_theme in filtered_df['anno1_theme'].unique(): 
        if ('kmeans' not in anno_1_theme.lower()) and ('unknown' not in anno_1_theme.lower()):
            theme_filtered_df  = filtered_df[(filtered_df['anno1_theme'] == anno_1_theme) &
                                            ~(filtered_df['anno2_theme'].str.contains('None')) & 
                                              ~(filtered_df['anno2_theme'].str.contains('Kmeans')) &
                                              ~(filtered_df['anno2_theme'].str.contains('Unknown'))]
            max_cosine_sims.append(theme_filtered_df.loc[(theme_filtered_df['average_sim'].idxmax())].to_dict())
 
res_df = pd.DataFrame(max_cosine_sims)

print("Synchronous Group Average Cosine Similarity")
print(f"Average Max Group Cosine Similarity: {res_df['average_sim'].mean():.2f}")
print(f"Standard Deviation of Group Cosine Similarity: { res_df['average_sim'].std():.2f}")

print(res_df.count())

Synchronous Group Average Cosine Similarity
Average Max Group Cosine Similarity: 0.43
Standard Deviation of Group Cosine Similarity: 0.08
anno1            25
anno2            25
anno1_theme      25
anno2_theme      25
average_sim      25
std_deviation    25
dtype: int64


### 4.c. Calculating Asynchronous Group Average Cosine Similarities

In [23]:
max_cosine_sims = []

for anno_1 , anno_2 in itertools.permutations(asynchronous_annotators , 2):
    filtered_df = global_average_df[(global_average_df['anno1'] == anno_1)&
                                    (global_average_df['anno2'] == anno_2)]
    for anno_1_theme in filtered_df['anno1_theme'].unique(): 
        if ('kmeans' not in anno_1_theme.lower()) and ('unknown' not in anno_1_theme.lower()):
            theme_filtered_df  = filtered_df[(filtered_df['anno1_theme'] == anno_1_theme) &
                                            ~(filtered_df['anno2_theme'].str.contains('None')) & 
                                              ~(filtered_df['anno2_theme'].str.contains('Kmeans')) &
                                              ~(filtered_df['anno2_theme'].str.contains('Unknown'))]
            max_cosine_sims.append(theme_filtered_df.loc[(theme_filtered_df['average_sim'].idxmax())].to_dict())
 
res_df = pd.DataFrame(max_cosine_sims)


print("Asynchronous Group Average Cosine Similarity")
print(f"Average Max Group Cosine Similarity: {res_df['average_sim'].mean():.2f}")
print(f"Standard Deviation of Group Cosine Similarity: { res_df['average_sim'].std():.2f}")

print(res_df.count())

Asynchronous Group Average Cosine Similarity
Average Max Group Cosine Similarity: 0.42
Standard Deviation of Group Cosine Similarity: 0.08
anno1            39
anno2            39
anno1_theme      39
anno2_theme      39
average_sim      39
std_deviation    39
dtype: int64


____

## 5. Getting top 25th percentile of closest vectors to each centroid

In [24]:
results = []

for annotator, df in annotator2df.items(): 
    cosine_sims = []
    themes = []
    for i , theme in enumerate(df['name'].unique()): 
        if ('kmeans' not in theme.lower()) and ('unknown' not in theme.lower()) and (theme != "None"): 
            result = {'annotator' : annotator}
            ids = df[df['name'] == theme]['tweet_id'].tolist()
            result['theme'] = theme
            result['tweets'] = df[df['name'] == theme]['text'].tolist()

            theme_vectors = sbert_vectors[ids]
            theme_centroid = np.average(theme_vectors, axis=0, keepdims=True)

            result['theme_centroid'] = theme_centroid
            result['theme_vectors'] = theme_vectors

            dot_prod = np.dot(theme_centroid , theme_vectors.T).squeeze(0)
            dot_prod = np.expand_dims(dot_prod , axis=-1)

            norm = np.linalg.norm(theme_vectors , axis=1, keepdims=True)
            cosine_sim = (dot_prod/norm)
            result['cosine_sim']  = cosine_sim.squeeze() 
            results.append(result)

In [25]:
quartile_map = {
    "0-25": (0, 25),
    "25-50": (25, 50),
    "50-75": (50, 75),
    "75-100": (75, 100),
}

quartile_results = {
    quartile: {'tweets': [],
              'vectors': [],
              'theme': [],
              'annotator': []}
    for quartile in quartile_map.keys()
}

for result in results:
    for quartile, (n, m) in quartile_map.items():
        low = np.percentile(result["cosine_sim"], n)
        high = np.percentile(result["cosine_sim"], m)
        
        quartile_indices = np.where((result["cosine_sim"] >= low) & (result["cosine_sim"] <= high))
        quartile_vectors = result["theme_vectors"][quartile_indices]
        quartile_tweets = np.array(result["tweets"])[quartile_indices]
        
        annotator = result["annotator"]
        theme = result["theme"]

        quartile_results[quartile]["tweets"].extend(quartile_tweets)
        quartile_results[quartile]["vectors"].extend(quartile_vectors)
        quartile_results[quartile]["theme"].extend([theme] * quartile_tweets.shape[0])
        quartile_results[quartile]["annotator"].extend([annotator] * quartile_tweets.shape[0])

quartile_dfs = {
    quartile: pd.DataFrame(data=quartile_data, columns=["theme", "annotator", "tweets"])
    for quartile, quartile_data in quartile_results.items()
}

### 4.a. Random Sampling rows for manual annotations

In [26]:
for quartile, df in quartile_dfs.items():
    # Get unique themes in this quartile
    # themes = df['theme'].unique()
    async_themes = df[df['annotator'].isin(asynchronous_annotators)]['theme'].unique()
    sync_themes = df[df['annotator'].isin(synchronous_annotators)]['theme'].unique()
    
    # Calculate samples per theme (200 total / number of themes)
    # samples_per_theme = 200 // len(themes)
    # remaining_samples = 200 % len(themes)
    async_samples_per_theme = 50 // len(async_themes)
    async_remaining_samples = 50 % len(async_themes)
    sync_samples_per_theme = 50 // len(sync_themes)
    sync_remaining_samples = 50 % len(sync_themes)
    
    # Sample from synchronous annotators
    sync_samples = []
    for i, theme in enumerate(sync_themes):
        theme_df = df[(df['annotator'].isin(synchronous_annotators)) & (df['theme'] == theme)]
        n_samples = sync_samples_per_theme + (1 if i < sync_remaining_samples else 0)
        n_samples = min(n_samples, len(theme_df))  # Don't sample more than available
        if n_samples > 0:
            sync_samples.append(theme_df.sample(n=n_samples))
    
    sync_quartile = pd.concat(sync_samples, ignore_index=True)[['theme', 'tweets']]
    
    # Sample from asynchronous annotators
    async_samples = []
    for i, theme in enumerate(async_themes):
        theme_df = df[(df['annotator'].isin(asynchronous_annotators)) & (df['theme'] == theme)]
        n_samples = async_samples_per_theme + (1 if i < async_remaining_samples else 0)
        n_samples = min(n_samples, len(theme_df))  # Don't sample more than available
        if n_samples > 0:
            async_samples.append(theme_df.sample(n=n_samples))
    
    async_quartile = pd.concat(async_samples, ignore_index=True)[['theme', 'tweets']]
    
    # Save to CSV
    sync_quartile.to_csv(f'./dataset/generated_samples/pacheco_sync_sample_{quartile}.csv', index=False)
    async_quartile.to_csv(f'./dataset/generated_samples/pacheco_async_sample_{quartile}.csv', index=False)
    print(len(sync_quartile), len(async_quartile))


50 50
50 50
50 50
50 50


____

## 5. Calculating Intra- and Inter-cluster Similarity

In [28]:
top_25_results = {}

for result in results: 

    top_25_results[result['theme']] = {}

    m = np.percentile(result['cosine_sim'] , 75)
    top_25_indices = np.where(result['cosine_sim']>=m)
    top_25_vectors = result['theme_vectors'][top_25_indices]
    top_25_tweets = np.array(result['tweets'])[top_25_indices]
    annotator=result['annotator']
    theme=result['theme']

    top_25_results[theme]['top_25_tweets'] = (top_25_tweets)
    top_25_results[theme]['top_25_vectors'] = (top_25_vectors )
    top_25_results[theme]['annotator'] = (annotator)

### 5.a. Calculating similarities for top 25th percentile clusters

In [29]:
from sklearn.metrics.pairwise import cosine_similarity

top_25_average_sims = []

for anno_1_theme , anno1_top_25 in tqdm(top_25_results.items()): 
    for anno_2_theme , anno2_top_25 in top_25_results.items(): 

        anno1_top_25_vectors = anno1_top_25['top_25_vectors']
        anno2_top_25_vectors = anno2_top_25['top_25_vectors']

        cosine_sims = cosine_similarity(anno1_top_25_vectors , anno2_top_25_vectors)
        average_sim = np.average(cosine_sims)
        std_deviation = np.std(cosine_sims)

        top_25_average_sims.append({'anno_1' : anno1_top_25['annotator'] , 
                                   'anno_2' : anno2_top_25['annotator'], 
                                   'anno_1_theme' : anno_1_theme,
                                   'anno_2_theme' : anno_2_theme,
                                   'average_sim' : average_sim , 
                                   'std_dev_sim' : std_deviation})


  0%|          | 0/84 [00:00<?, ?it/s]

In [30]:
top_25_average_df = pd.DataFrame(top_25_average_sims)
top_25_average_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,average_sim,std_dev_sim
0,async_1,async_1,AdvocateForGreenEnergy,AdvocateForGreenEnergy,0.626497,0.093036
1,async_1,async_1,AdvocateForGreenEnergy,GreenPowerExpensive,0.404321,0.082457
2,async_1,async_1,AdvocateForGreenEnergy,ClimateActionDonation,0.540822,0.092856
3,async_1,async_1,AdvocateForGreenEnergy,GreenEnergyWorks,0.423361,0.098803
4,async_1,async_1,AdvocateForGreenEnergy,AdvocateForGreenEnergyPoliticalAction,0.612084,0.074921
...,...,...,...,...,...,...
7051,async_2,async_2,EconomicAdvantagesOfNuclearEnergy,SupportForBuildBackBetterAct,0.208997,0.034858
7052,async_2,async_2,EconomicAdvantagesOfNuclearEnergy,SupportForOilAndGas,0.181971,0.085022
7053,async_2,async_2,EconomicAdvantagesOfNuclearEnergy,EconomicAdvantagesOfOilAndGas,0.149216,0.080309
7054,async_2,async_2,EconomicAdvantagesOfNuclearEnergy,InroadsMadeByNuclearEnergy,0.228152,0.073981


In [31]:
top_25_intra_cluster_df = top_25_average_df[(top_25_average_df['anno_1']==top_25_average_df['anno_2']) & (top_25_average_df['anno_1_theme']==top_25_average_df['anno_2_theme'])] 

### 5.b. Intra- and Inter-theme similarities for top 25th percentile subsets in synchronous and asynchronous experiments

In [32]:
print(f"Synchronous top 25% intra theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous top 25% intra theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous top 25% intra theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous top 25% intra theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous top 25% intra theme similarity average: 0.66
Synchronous top 25% intra theme similarity standard deviation: 0.11
Asynchronous top 25% intra theme similarity average: 0.65
Asynchronous top 25% intra theme similarity standard deviation: 0.10


In [33]:
top_25_intra_cluster_df = top_25_average_df[(top_25_average_df['anno_1']==top_25_average_df['anno_2']) & (top_25_average_df['anno_1_theme']!=top_25_average_df['anno_2_theme'])] 

In [34]:
print(f"Synchronous top 25% inter theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous top 25% inter theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous top 25% inter theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous top 25% inter theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous top 25% inter theme similarity average: 0.39
Synchronous top 25% inter theme similarity standard deviation: 0.10
Asynchronous top 25% inter theme similarity average: 0.39
Asynchronous top 25% inter theme similarity standard deviation: 0.11


### 5.c. Intra- and Inter-theme similarities for whole set in synchronous and asynchronous experiments

In [35]:
global_average_df = pd.DataFrame(global_average_sims)

intra_global_average_df = global_average_df[(global_average_df['anno1']==global_average_df['anno2']) & (global_average_df['anno1_theme']==global_average_df['anno2_theme'])] 


print(f"Synchronous global intra theme similarity average: {intra_global_average_df[intra_global_average_df['anno1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous global intra theme similarity standard deviation: {intra_global_average_df[intra_global_average_df['anno1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous global intra theme similarity average: {intra_global_average_df[intra_global_average_df['anno1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous global intra theme similarity standard deviation: {intra_global_average_df[intra_global_average_df['anno1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous global intra theme similarity average: 0.44
Synchronous global intra theme similarity standard deviation: 0.10
Asynchronous global intra theme similarity average: 0.43
Asynchronous global intra theme similarity standard deviation: 0.09


In [36]:
inter_global_avg_df = global_average_df[(global_average_df['anno1']==global_average_df['anno2']) & (global_average_df['anno1_theme']!=global_average_df['anno2_theme'])] 

print(f"Synchronous global inter theme similarity average: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous global inter theme similarity standard deviation: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous global inter theme similarity average: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous global inter theme similarity standard deviation: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous global inter theme similarity average: 0.30
Synchronous global inter theme similarity standard deviation: 0.07
Asynchronous global inter theme similarity average: 0.29
Asynchronous global inter theme similarity standard deviation: 0.07
